# Module 5 — Cost Drag Analysis

Exploratory notebook for `analysis/costs.py`. This is the core Boglehead
argument: expense ratios compound just like returns do, but against you. A
1% expense ratio doesn't just cost 1% a year — it costs 1% *of a growing
balance*, every year, forever.

**Key concepts:** compound growth math, NumPy parameter sweeps over expense
ratios, clear comparative charts with `matplotlib`.

> Convention (see `SPEC.md`): explore here first, then refactor the working
> logic into `analysis/costs.py`. This notebook is kept as an honest
> artifact of the process.

## Setup

In [1]:
import sys
from pathlib import Path

# Make the project root importable when running from notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from analysis import fetch, portfolio, costs
from analysis.config import (
    DEFAULT_TICKERS,
    DEFAULT_WEIGHTS,
    DEFAULT_START,
    DEFAULT_INVESTMENT,
    DEFAULT_EXPENSE_RATIOS,
)

# Pull from the local cache written by Module 1 (network only on a miss).
prices = fetch.load_or_fetch(DEFAULT_TICKERS, DEFAULT_START)
built = portfolio.build_portfolio(prices, DEFAULT_WEIGHTS, DEFAULT_INVESTMENT)
built["total"].tail()

Date
2026-07-31    46104.877075
2026-08-03    46712.746232
2026-08-04    47551.559625
2026-08-05    47440.178255
2026-08-06    47360.545967
Name: total, dtype: float64

## 1. Applying an expense ratio to real portfolio history

`apply_expense_ratio` spreads the annual ratio evenly across trading days —
each day's growth factor is multiplied by `(1 - expense_ratio / 252)` on top
of that day's actual price return, then compounded forward. Compare the
blended expense ratio of the actual 60/30/10 holdings (all under 0.1%)
against a hypothetical 1% actively-managed fund tracking the same returns.

In [2]:
blended_er = sum(DEFAULT_WEIGHTS[t] * DEFAULT_EXPENSE_RATIOS[t] for t in DEFAULT_TICKERS)
print(f"Blended expense ratio of the 60/30/10 mix: {blended_er:.4%}")

scenarios = costs.compare_funds(built["total"], {"actual (low-cost)": blended_er, "1.00% active fund": 0.01})
scenarios.tail()

Blended expense ratio of the 60/30/10 mix: 0.0420%


,actual (low-cost),1.00% active fund
Date,,
2026-07-31,45843.285042,40263.278868
2026-08-03,46447.627829,40792.510735
2026-08-04,47281.601730,41523.367577
2026-08-05,47170.774072,41424.462320
2026-08-06,47091.515515,41353.286843


In [3]:
scenarios.plot(figsize=(10, 5), title="Same returns, different expense ratios")
plt.ylabel("Value ($)")
plt.tight_layout()
plt.savefig("images/actual_vs_active_fund.png", dpi=110, bbox_inches="tight")
plt.close()

![Same returns, different expense ratios](images/actual_vs_active_fund.png)

In [4]:
terminal = scenarios.iloc[-1]
gap = terminal["actual (low-cost)"] - terminal["1.00% active fund"]
print(terminal.map(lambda x: f"${x:,.2f}"))
print(f"\nCost of the higher expense ratio over this window: ${gap:,.2f}")

actual (low-cost)    $47,091.52
1.00% active fund    $41,353.29
Name: 2026-08-06 00:00:00, dtype: object

Cost of the higher expense ratio over this window: $5,738.23


## 2. Pure-math projection — 30 years, a sweep of expense ratios

`cost_drag_over_time` doesn't need price data at all: it projects a constant
assumed annual return forward under different expense ratios and shows the
terminal-value curve. Sweep from 0% to 2% with `np.linspace` to see where the
curves start to visibly separate.

In [5]:
expense_ratio_sweep = np.linspace(0.0, 0.02, 21)
projection = costs.cost_drag_over_time(
    initial_investment=10_000.0,
    annual_return=0.07,
    years=30,
    expense_ratios=expense_ratio_sweep,
)
projection.tail()

,0.000,0.001,0.002,0.003,0.004,0.005,0.006,0.007,0.008,0.009,...,0.011,0.012,0.013,0.014,0.015,0.016,0.017,0.018,0.019,0.020
year,,,,,,,,,,,,,,,,,,,,,
26,58073.529249,56582.341259,55128.007157,53709.651685,52326.419526,50977.474868,49662.000979,48379.199790,47128.291486,45908.514105,...,43559.391205,42428.607555,41326.077825,40251.123618,39203.082161,38181.305963,37185.162478,36214.033773,35267.316204,34344.420103
27,62138.676297,60482.562042,58868.993722,57296.919321,55765.311817,54273.168618,52819.511001,51403.383569,50023.853715,48680.011102,...,46095.854555,44853.826763,43644.057531,42465.740439,41318.088443,40200.333422,39111.725746,38051.533847,37019.043800,36013.558920
28,66488.383638,64651.625044,62863.843636,61123.780562,59430.208109,57781.928969,56177.775511,54616.609076,53097.319287,51618.823372,...,48780.016166,47417.671501,46092.052718,44802.205478,43547.199315,42326.127054,41138.104257,39982.268674,38857.779706,37763.817884
29,71142.570492,69108.061558,67129.784066,65206.237866,63335.961386,61517.530677,59749.558478,58030.693309,56359.618584,54735.051739,...,51620.476507,50128.065604,48677.355955,47267.222824,45896.570718,44564.332652,43269.469439,42010.968987,40787.845624,39599.139433
30,76122.550427,73871.680241,71685.211212,69561.362493,67498.400769,65494.639035,63548.435406,61658.191948,59822.353550,58039.406813,...,54626.336855,52993.385834,51407.668850,49867.865423,48372.690708,46920.894563,45511.260650,44142.605553,42813.777916,41523.657609


In [6]:
terminal_values = projection.loc[30]
plt.figure(figsize=(10, 5))
plt.plot(terminal_values.index * 100, terminal_values.values)
plt.xlabel("Expense ratio (%)")
plt.ylabel("Terminal value after 30 years ($)")
plt.title("$10,000 at 7%/yr for 30 years — terminal value vs. expense ratio")
plt.tight_layout()
plt.savefig("images/terminal_value_vs_expense_ratio.png", dpi=110, bbox_inches="tight")
plt.close()

![Terminal value after 30 years vs. expense ratio](images/terminal_value_vs_expense_ratio.png)

## 3. The headline comparison — 0.03% vs. 1.00%

The classic Boglehead talking point: `$10,000` invested for 30 years at a 7%
annual return, comparing a typical Vanguard index fund expense ratio (0.03%)
to a typical actively managed fund (1.00%).

In [7]:
headline = costs.cost_drag_over_time(10_000.0, 0.07, 30, [0.0003, 0.01])
headline_terminal = headline.loc[30]
dollar_gap = headline_terminal[0.0003] - headline_terminal[0.01]

print(f"0.03% ER terminal value: ${headline_terminal[0.0003]:,.2f}")
print(f"1.00% ER terminal value: ${headline_terminal[0.01]:,.2f}")
print(f"Dollar difference over 30 years: ${dollar_gap:,.2f}")

0.03% ER terminal value: $75,440.42
1.00% ER terminal value: $56,307.88
Dollar difference over 30 years: $19,132.54


In [8]:
headline.plot(figsize=(10, 5), title="$10,000 at 7%/yr — 0.03% vs. 1.00% expense ratio")
plt.ylabel("Value ($)")
plt.xlabel("Year")
plt.tight_layout()
plt.savefig("images/headline_003_vs_100.png", dpi=110, bbox_inches="tight")
plt.close()

![$10,000 at 7%/yr — 0.03% vs. 1.00% expense ratio](images/headline_003_vs_100.png)

## 4. When do the curves start to visibly separate?

Early on, the dollar gap is small — compounding needs time to work. Track
the *percentage* gap between the two scenarios year over year to see when it
becomes hard to ignore.

In [9]:
pct_gap = (headline[0.0003] - headline[0.01]) / headline[0.0003]
pct_gap.iloc[1:].plot(figsize=(10, 4), title="Cost drag as % of terminal value, by year", color="firebrick")
plt.ylabel("% difference")
plt.xlabel("Year")
plt.tight_layout()
plt.savefig("images/cost_drag_pct_gap.png", dpi=110, bbox_inches="tight")
plt.close()

![Cost drag as % of terminal value, by year](images/cost_drag_pct_gap.png)

In [10]:
first_5pct_year = pct_gap[pct_gap >= 0.05].index.min()
print(f"Gap first crosses 5% of terminal value in year {first_5pct_year}")

Gap first crosses 5% of terminal value in year 6


## Checkpoint

**How does compounding make a 1% expense ratio so costly over 30 years?**

A 1% expense ratio isn't a flat $100-a-year fee on the original $10,000 — it's
1% *of whatever the balance has grown to that year*, deducted before the next
year's growth compounds. Early on the dollar amount is small because the
balance is still close to $10,000, but the fee is also compounding: money
that would have earned 7% next year, and every year after, is gone instead of
growing. Over 30 years that lost compounding is the majority of the gap —
`cost_drag_over_time` shows the terminal-value difference is far larger than
`30 years x 0.97% x $10,000` would suggest, because the *base* the fee is
skimming from would itself have kept growing.